# 🛰️ Spectrum-SLM — 176 Patches (patch_size = 1) — Kaggle GPU Training

**Authors:** Anjani, Ashish Joshi, Mayank  
**Guide:** Dr. Abhinandan S.P.

## ⚙️ Before running:
1. Go to **Settings (right panel) → Accelerator → GPU T4 x2**
2. Go to **Settings → Internet → Turn ON**
3. Click **Run All** from the top menu

## What this notebook does:
| Phase | Task | Time estimate |
|-------|------|---------------|
| **1** | Clone repo from GitHub | ~30 sec |
| **2** | Install dependencies | ~1 min |
| **3** | Verify GPU + config | ~10 sec |
| **4** | Phase 1 — Masked Spectrum Modelling (pre-training) | ~5–10 min |
| **5** | Phase 2 — Supervised Multi-task Fine-tuning | ~15–30 min |
| **6** | Evaluation & metrics | ~2 min |
| **7** | Save checkpoints to /kaggle/working | instant |

---
## Cell 1 — Clone the latest code from GitHub

In [ ]:
# Clone the Spectrum-SLM repo (pinned to master with 176-patch changes)
!git clone https://github.com/mayanktewatia10/Spectrum-SLM.git /kaggle/working/Spectrum-SLM
%cd /kaggle/working/Spectrum-SLM
!git log --oneline -5

---
## Cell 2 — Install dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy torch torchvision
print("✅ Dependencies installed")

---
## Cell 3 — Verify GPU & print config

In [ ]:
import torch
import sys
import os

print("=" * 55)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.version.cuda}")
print(f"  GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")
print(f"  GPU Mem : {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB" if torch.cuda.is_available() else "")
print("=" * 55)

# Add repo to path
sys.path.insert(0, '/kaggle/working/Spectrum-SLM')

# Print config values
import config as cfg
print(f"\n  N_BINS     : {cfg.N_BINS}")
print(f"  PATCH_SIZE : {cfg.PATCH_SIZE}  ← should be 1")
print(f"  N_PATCHES  : {cfg.N_PATCHES}  ← should be 176")
print(f"  D_MODEL    : {cfg.D_MODEL}")
print(f"  N_HEAD     : {cfg.N_HEAD}")
print(f"  NUM_LAYERS : {cfg.NUM_LAYERS}")
print("\n✅ Config verified — 176 patches active")

---
## Cell 4 — Load & explore the dataset

> **Option A (recommended):** Upload your `.pth` / `.csv` files as a Kaggle Dataset and attach it via *Add Data*.  
> **Option B (auto):** The notebook will use synthetic data if no real files are found — perfect for a quick test.

In [ ]:
import os, glob

# ── Detect what data is available ────────────────────────────────────────────
KAGGLE_INPUT = '/kaggle/input'
LOCAL_DATA   = '/kaggle/working/Spectrum-SLM/Secondary_User/Symbol1_Modulation'

USE_SYNTHETIC = False
DATA_DIR      = None

# 1. Check for attached Kaggle Dataset (pth files)
pth_files = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*.pth'), recursive=True)
csv_files = glob.glob(os.path.join(KAGGLE_INPUT, '**', '*.csv'), recursive=True)

if pth_files:
    DATA_DIR = os.path.dirname(pth_files[0])
    print(f"✅ Found {len(pth_files)} .pth files in /kaggle/input")
    print(f"   DATA_DIR = {DATA_DIR}")
elif csv_files:
    DATA_DIR = os.path.dirname(csv_files[0])
    print(f"✅ Found {len(csv_files)} .csv files in /kaggle/input")
    print(f"   DATA_DIR = {DATA_DIR}")
elif os.path.exists(LOCAL_DATA):
    DATA_DIR = LOCAL_DATA
    print(f"✅ Using repo-bundled CSV data: {DATA_DIR}")
else:
    USE_SYNTHETIC = True
    print("⚠️  No real data found — switching to SYNTHETIC mode (10 000 samples)")
    print("   (Add your dataset via Kaggle 'Add Data' for real training)")

print(f"\n  Mode      : {'SYNTHETIC' if USE_SYNTHETIC else 'REAL DATA'}")
print(f"  Data dir  : {DATA_DIR or 'N/A (synthetic)'}")

---
## Cell 5 — Build DataLoaders

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/Spectrum-SLM')

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import config as cfg
from spectrum_slm_dataset import (
    generate_synthetic_psd, build_dataloaders,
    SpectrumNormalizer, SpectrumDataset, SpectrumAugmenter,
    N_BINS, N_PATCHES, PATCH_SIZE
)

BATCH_SIZE   = 64
NUM_WORKERS  = 2   # Kaggle supports multi-worker loading

if USE_SYNTHETIC:
    print("Generating 10 000 synthetic PSD samples...")
    psds, pu, mod, snr = generate_synthetic_psd(n_samples=10000, seed=42)
    normalizer = SpectrumNormalizer()
    psds_norm  = normalizer.fit_transform(psds)

    idx = np.arange(len(psds_norm))
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=pu, random_state=42)
    idx_v,  idx_te  = train_test_split(idx_tmp, test_size=0.50, stratify=pu[idx_tmp], random_state=42)

    aug = SpectrumAugmenter()
    tr_ds = SpectrumDataset(psds_norm[idx_tr], pu[idx_tr], mod[idx_tr], snr[idx_tr], phase=1, augmenter=aug)
    vl_ds = SpectrumDataset(psds_norm[idx_v],  pu[idx_v],  mod[idx_v],  snr[idx_v],  phase=1)
    te_ds = SpectrumDataset(psds_norm[idx_te], pu[idx_te], mod[idx_te], snr[idx_te], phase=2)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    pu_counts  = np.bincount(pu[idx_tr])
    pu_weights = torch.tensor(len(idx_tr) / (2 * np.maximum(pu_counts, 1)), dtype=torch.float32)
    print(f"  Train: {len(idx_tr)} | Val: {len(idx_v)} | Test: {len(idx_te)}")
else:
    tr_loader, vl_loader, te_loader, normalizer, meta = build_dataloaders(
        data_dir    = DATA_DIR,
        phase       = 1,
        batch_size  = BATCH_SIZE,
        num_workers = NUM_WORKERS,
    )
    pu_weights = meta['pu_weights']

print(f"\n✅ DataLoaders ready: {len(tr_loader)} train batches | batch size {BATCH_SIZE}")
print(f"   Each PSD = {N_BINS} bins → {N_PATCHES} patches of size {PATCH_SIZE}")

---
## Cell 6 — Build Spectrum-SLM model (176 patches)

In [ ]:
from spectrum_slm_model import SpectrumSLM, MultiTaskLoss, MSMLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

model = SpectrumSLM(
    n_bins          = cfg.N_BINS,       # 176
    patch_size      = cfg.PATCH_SIZE,   # 1  ← 176 patches
    d_model         = cfg.D_MODEL,      # 128
    nhead           = cfg.N_HEAD,       # 4
    num_layers      = cfg.NUM_LAYERS,   # 4
    dim_feedforward = cfg.DIM_FEEDFORWARD,  # 512
    dropout         = cfg.DROPOUT,      # 0.1
    n_mod_classes   = 4,
)

total_params = model.count_parameters()
print(f"  Total trainable parameters : {total_params:,}")
print(f"  Sequence length in transformer: 177  (176 patches + 1 CLS token)")

# Quick shape sanity check
dummy = torch.randn(2, 176)
out   = model(dummy)
print(f"\n  Forward pass shapes:")
for k, v in out.items():
    print(f"    {k:12s}: {tuple(v.shape)}")
print("\n✅ Model built and forward pass OK")

---
## Cell 7 — Phase 1: Masked Spectrum Modelling (self-supervised pre-training)

In [ ]:
import time
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

PHASE1_EPOCHS  = 30
PHASE1_LR      = 3e-4
PHASE1_PATIENCE = 6
SAVE_DIR = '/kaggle/working/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

model = model.to(device)
optimizer1 = optim.AdamW(model.parameters(), lr=PHASE1_LR, weight_decay=1e-4)
scheduler1 = CosineAnnealingLR(optimizer1, T_max=PHASE1_EPOCHS, eta_min=1e-5)
msm_loss   = MSMLoss()

best_val  = float('inf')
patience  = PHASE1_PATIENCE
history_p1 = []

print(f"{'='*60}")
print(f"  PHASE 1 — Masked Spectrum Modelling ({PHASE1_EPOCHS} epochs max)")
print(f"  Patch size: {PATCH_SIZE}  |  Patches per sample: {N_PATCHES}")
print(f"{'='*60}")

for epoch in range(1, PHASE1_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    t0 = time.time()
    tr_losses = []

    for psd, mask in tr_loader:
        psd, mask = psd.to(device), mask.to(device)
        # Ground truth: reshape (B, 176) → (B, 176, 1)
        true_patches = psd.view(-1, N_PATCHES, PATCH_SIZE)

        optimizer1.zero_grad()
        out  = model(psd, mask=mask, return_msm=True)
        loss = msm_loss(out['msm_pred'], true_patches, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer1.step()
        tr_losses.append(loss.item())

    # ── Validation ──────────────────────────────────────────────────────────
    model.eval()
    vl_losses = []
    with torch.no_grad():
        for psd, mask in vl_loader:
            psd, mask = psd.to(device), mask.to(device)
            true_patches = psd.view(-1, N_PATCHES, PATCH_SIZE)
            out  = model(psd, mask=mask, return_msm=True)
            loss = msm_loss(out['msm_pred'], true_patches, mask)
            vl_losses.append(loss.item())

    scheduler1.step()
    tr_l  = np.mean(tr_losses)
    vl_l  = np.mean(vl_losses)
    elapsed = time.time() - t0
    history_p1.append({'epoch': epoch, 'train': tr_l, 'val': vl_l})

    print(f"  Ep {epoch:3d}/{PHASE1_EPOCHS} | Train: {tr_l:.4f} | Val: {vl_l:.4f} | {elapsed:.1f}s")

    if vl_l < best_val:
        best_val = vl_l
        patience = PHASE1_PATIENCE
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer1.state_dict(), 'val_loss': vl_l},
                   os.path.join(SAVE_DIR, 'slm_phase1_best.pt'))
        print(f"      ✓ Best checkpoint saved (val={vl_l:.4f})")
    else:
        patience -= 1
        if patience == 0:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

print(f"\n✅ Phase 1 done. Best val MSM loss: {best_val:.4f}")

---
## Cell 8 — Phase 2: Supervised Multi-task Fine-tuning

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

PHASE2_EPOCHS   = 50
PHASE2_LR       = 1e-4
PHASE2_PATIENCE = 8

# Load best Phase 1 weights
p1_ckpt = os.path.join(SAVE_DIR, 'slm_phase1_best.pt')
if os.path.exists(p1_ckpt):
    ckpt = torch.load(p1_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f"  ✓ Loaded Phase 1 checkpoint (epoch {ckpt['epoch']})")
else:
    print("  ⚠️  No Phase 1 checkpoint found, using current weights")

# Rebuild loaders for Phase 2 (returns labels)
if USE_SYNTHETIC:
    for ds in [tr_ds, vl_ds, te_ds]:
        ds.phase = 2
else:
    tr_loader, vl_loader, te_loader, normalizer, meta = build_dataloaders(
        data_dir    = DATA_DIR,
        phase       = 2,
        batch_size  = BATCH_SIZE,
        num_workers = NUM_WORKERS,
    )
    pu_weights = meta['pu_weights']

criterion2 = MultiTaskLoss(
    alpha=1.0, beta=0.5, gamma=0.3,
    pu_class_weight=pu_weights.to(device),
    focal_gamma=2.0,
    learn_weights=True,
).to(device)

optimizer2 = optim.AdamW(
    list(model.parameters()) + list(criterion2.parameters()),
    lr=PHASE2_LR, weight_decay=1e-4
)
scheduler2 = OneCycleLR(
    optimizer2, max_lr=PHASE2_LR * 10,
    epochs=PHASE2_EPOCHS, steps_per_epoch=len(tr_loader),
    pct_start=0.1, anneal_strategy='cos',
)

best_val2 = float('inf')
patience2 = PHASE2_PATIENCE
history_p2 = []

print(f"{'='*60}")
print(f"  PHASE 2 — Supervised Fine-tuning ({PHASE2_EPOCHS} epochs max)")
print(f"  Kendall uncertainty weighting: ON")
print(f"{'='*60}")

for epoch in range(1, PHASE2_EPOCHS + 1):
    model.train()
    t0 = time.time()
    tr_t, tr_pu, tr_mod, tr_snr = [], [], [], []

    for psd, pu_lab, mod_lab, snr_lab in tr_loader:
        psd    = psd.to(device)
        pu_lab = pu_lab.to(device)
        mod_lab = mod_lab.to(device)
        snr_lab = snr_lab.to(device)
        valid = mod_lab >= 0
        if valid.sum() == 0:
            continue

        optimizer2.zero_grad()
        out = model(psd)
        total, bd = criterion2(
            out['pu_logits'][valid],  pu_lab[valid],
            out['mod_logits'][valid], mod_lab[valid],
            out['snr_pred'][valid],   snr_lab[valid],
        )
        total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer2.step()
        scheduler2.step()
        tr_t.append(bd['total']); tr_pu.append(bd['pu'])
        tr_mod.append(bd['mod']); tr_snr.append(bd['snr'])

    model.eval()
    vl_t = []
    with torch.no_grad():
        for psd, pu_lab, mod_lab, snr_lab in vl_loader:
            psd    = psd.to(device)
            pu_lab = pu_lab.to(device)
            mod_lab = mod_lab.to(device)
            snr_lab = snr_lab.to(device)
            valid = mod_lab >= 0
            if valid.sum() == 0:
                continue
            out   = model(psd)
            loss, _ = criterion2(
                out['pu_logits'][valid],  pu_lab[valid],
                out['mod_logits'][valid], mod_lab[valid],
                out['snr_pred'][valid],   snr_lab[valid],
            )
            vl_t.append(loss.item())

    elapsed = time.time() - t0
    vl_mean = np.mean(vl_t) if vl_t else float('inf')
    history_p2.append({'epoch': epoch, 'train': np.mean(tr_t), 'val': vl_mean})

    print(f"  Ep {epoch:3d}/{PHASE2_EPOCHS} | "
          f"Train: {np.mean(tr_t):.4f} "
          f"(PU={np.mean(tr_pu):.3f} Mod={np.mean(tr_mod):.3f} SNR={np.mean(tr_snr):.3f}) "
          f"| Val: {vl_mean:.4f} | {elapsed:.1f}s")

    if vl_mean < best_val2:
        best_val2 = vl_mean
        patience2 = PHASE2_PATIENCE
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'optimizer': optimizer2.state_dict(), 'val_loss': vl_mean},
                   os.path.join(SAVE_DIR, 'slm_phase2_best.pt'))
        print(f"      ✓ Best checkpoint saved (val={vl_mean:.4f})")
    else:
        patience2 -= 1
        if patience2 == 0:
            print(f"  ⏹ Early stopping at epoch {epoch}")
            break

print(f"\n✅ Phase 2 done. Best val loss: {best_val2:.4f}")

---
## Cell 9 — Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_absolute_error
import json

# Load best Phase 2 weights
p2_ckpt = os.path.join(SAVE_DIR, 'slm_phase2_best.pt')
if os.path.exists(p2_ckpt):
    ckpt = torch.load(p2_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f"  ✓ Loaded Phase 2 checkpoint (epoch {ckpt['epoch']})")

model.eval().to(device)
if USE_SYNTHETIC:
    te_ds.phase = 2

all_pu_pred, all_pu_true   = [], []
all_mod_pred, all_mod_true = [], []
all_snr_pred, all_snr_true = [], []

with torch.no_grad():
    for psd, pu_lab, mod_lab, snr_lab in te_loader:
        psd = psd.to(device)
        out = model(psd)
        all_pu_pred.extend(out['pu_logits'].argmax(1).cpu().numpy())
        all_pu_true.extend(pu_lab.numpy())
        all_mod_pred.extend(out['mod_logits'].argmax(1).cpu().numpy())
        all_mod_true.extend(mod_lab.numpy())
        all_snr_pred.extend(out['snr_pred'].cpu().numpy())
        all_snr_true.extend(snr_lab.numpy())

all_pu_pred  = np.array(all_pu_pred)
all_pu_true  = np.array(all_pu_true)
all_mod_pred = np.array(all_mod_pred)
all_mod_true = np.array(all_mod_true)
all_snr_pred = np.array(all_snr_pred)
all_snr_true = np.array(all_snr_true)

pu_acc  = accuracy_score(all_pu_true, all_pu_pred)
pu_f1   = f1_score(all_pu_true, all_pu_pred, average='binary', zero_division=0)
try:
    pu_auc = roc_auc_score(all_pu_true, all_pu_pred)
except:
    pu_auc = float('nan')

valid_mod = all_mod_true >= 0
if valid_mod.sum() > 0:
    mod_acc = accuracy_score(all_mod_true[valid_mod], all_mod_pred[valid_mod])
    mod_f1  = f1_score(all_mod_true[valid_mod], all_mod_pred[valid_mod], average='macro', zero_division=0)
else:
    mod_acc = mod_f1 = float('nan')

snr_mae  = mean_absolute_error(all_snr_true, all_snr_pred)
snr_rmse = float(np.sqrt(np.mean((all_snr_pred - all_snr_true)**2)))

metrics = {
    'patch_size'    : PATCH_SIZE,
    'n_patches'     : N_PATCHES,
    'pu_accuracy'   : float(pu_acc),
    'pu_f1'         : float(pu_f1),
    'pu_auc'        : float(pu_auc),
    'mod_accuracy'  : float(mod_acc),
    'mod_f1_macro'  : float(mod_f1),
    'snr_mae_db'    : float(snr_mae),
    'snr_rmse_db'   : float(snr_rmse),
    'n_test_samples': int(all_pu_pred.shape[0]),
}

print(f"\n{'='*55}")
print(f"  EVALUATION — Spectrum-SLM (176 patches, patch_size=1)")
print(f"{'='*55}")
print(f"  PU Detection  — Acc: {pu_acc*100:.2f}%  F1: {pu_f1:.4f}  AUC: {pu_auc:.4f}")
print(f"  Modulation    — Acc: {mod_acc*100:.2f}%  F1-macro: {mod_f1:.4f}")
print(f"  SNR Estimation— MAE: {snr_mae:.3f} dB   RMSE: {snr_rmse:.3f} dB")
print(f"  Test samples  : {metrics['n_test_samples']:,}")
print(f"{'='*55}")

with open(os.path.join(SAVE_DIR, 'metrics_176patches.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n  ✓ Metrics saved → {SAVE_DIR}/metrics_176patches.json")

---
## Cell 10 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spectrum-SLM — 176 Patches (patch_size=1)', color='white', fontsize=14, fontweight='bold')

# Phase 1
if history_p1:
    ep1 = [h['epoch'] for h in history_p1]
    axes[0].plot(ep1, [h['train'] for h in history_p1], label='Train MSM Loss', color='#58a6ff')
    axes[0].plot(ep1, [h['val']   for h in history_p1], label='Val MSM Loss',   color='#3fb950', linestyle='--')
    axes[0].set_title('Phase 1 — Masked Spectrum Modelling', color='white')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

# Phase 2
if history_p2:
    ep2 = [h['epoch'] for h in history_p2]
    axes[1].plot(ep2, [h['train'] for h in history_p2], label='Train Loss', color='#ffa657')
    axes[1].plot(ep2, [h['val']   for h in history_p2], label='Val Loss',   color='#f78166', linestyle='--')
    axes[1].set_title('Phase 2 — Supervised Fine-tuning', color='white')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Multi-task Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, 'training_curves_176patches.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  ✓ Plot saved → {plot_path}")

---
## Cell 11 — List all saved output files

In [ ]:
import os
print("Files in /kaggle/working/checkpoints:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:45s}  {size/1024:.1f} KB")

print("\n✅ All done! Download files from Output tab on the right.")